# Capstone Step 5: Data Wrangling & Exploration

In this notebook, we will demonstrate the extraction, cleaning, and exploration of food nutritional data. To achieve excellence, we will pull data from **multiple disparate sources**:
1. A local `.csv` file containing simulated product data (which includes intentional errors, duplicates, and missing values).
2. The live **Open Food Facts API** to fetch real-world data dynamically.

We will merge these sources, clean the anomalies, and explore the dataset visually.

In [ ]:
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme(style="whitegrid")

### 1. Data Extraction (Multiple Sources)

In [ ]:
# Source A: Local CSV File (Messy Data)
df_local = pd.read_csv('local_products.csv')
print("Local CSV Data (Before Cleaning):")
display(df_local)

In [ ]:
# Source B: Open Food Facts API (Live Data)
def fetch_product(barcode):
    url = f"https://world.openfoodfacts.org/api/v0/product/{barcode}.json"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        if data.get('status') == 1:
            p = data['product']
            nut = p.get('nutriments', {})
            return {
                'barcode': barcode,
                'product_name': p.get('product_name', 'Unknown'),
                'energy_100g': nut.get('energy-kj_100g'),
                'proteins_100g': nut.get('proteins_100g'),
                'carbohydrates_100g': nut.get('carbohydrates_100g'),
                'sugars_100g': nut.get('sugars_100g'),
                'fat_100g': nut.get('fat_100g'),
                'fiber_100g': nut.get('fiber_100g'),
                'ingredients_text': p.get('ingredients_text_en', '')
            }
    return None

# Fetch some common products (Nutella, Coke, Oreos, Fairlife Milk)
barcodes_to_fetch = ['3017620422003', '5449000000996', '0044000032029', '0811620020617']
api_products = [fetch_product(bc) for bc in barcodes_to_fetch]
api_products = [p for p in api_products if p is not None]

df_api = pd.DataFrame(api_products)
print("API Data:")
display(df_api)

### 2. Merging & Data Wrangling

In [ ]:
# Merge both sources
df_merged = pd.concat([df_local, df_api], ignore_index=True)

# Step 1: Remove Exact Duplicates
print(f"Rows before dropping duplicates: {len(df_merged)}")
df_cleaned = df_merged.drop_duplicates()
print(f"Rows after dropping duplicates: {len(df_cleaned)}")

In [ ]:
# Step 2: Handling Missing Values (Imputation Strategy)
print("\nMissing values per column:")
print(df_cleaned.isnull().sum())

# If energy or core macros are entirely missing, the row is useless for a nutrition scorer. We drop those rows.
df_cleaned = df_cleaned.dropna(subset=['energy_100g', 'proteins_100g'])

# For fiber or sugars, a missing value often implies 0g in labeling databases.
df_cleaned['fiber_100g'] = df_cleaned['fiber_100g'].fillna(0.0)
df_cleaned['sugars_100g'] = df_cleaned['sugars_100g'].fillna(0.0)

print("\nMissing values after imputation:")
print(df_cleaned.isnull().sum())

### 3. Outlier Detection & Treatment
In food data, values per 100g cannot exceed 100g physically. Also, pure fat is ~3700 kJ per 100g, so energy cannot realistically exceed ~4000 kJ per 100g.

In [ ]:
def handle_outliers(df):
    df_filtered = df.copy()
    
    # Cap macros at 100g
    macro_cols = ['proteins_100g', 'carbohydrates_100g', 'sugars_100g', 'fat_100g', 'fiber_100g']
    for col in macro_cols:
        # Values cannot be negative or over 100g
        df_filtered = df_filtered[(df_filtered[col] >= 0) & (df_filtered[col] <= 100)]
        
    # Cap Energy at 4000 kJ (Approx 950 kcal)
    df_filtered = df_filtered[(df_filtered['energy_100g'] >= 0) & (df_filtered['energy_100g'] <= 4000)]
    
    return df_filtered

print(f"Rows before outlier filtering: {len(df_cleaned)}")
df_final = handle_outliers(df_cleaned)
print(f"Rows after outlier filtering: {len(df_final)}")
display(df_final)

### 4. Exploratory Data Analysis & Visualizations

In [ ]:
# Visualization 1: Macro Nutrient Distribution
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_final[['proteins_100g', 'carbohydrates_100g', 'sugars_100g', 'fat_100g', 'fiber_100g']])
plt.title('Distribution of Macronutrients per 100g', fontsize=16)
plt.ylabel('Grams (g)')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Visualization 2: Correlation Heatmap
# This helps us see relationships (e.g., do high sugar foods also have high carbs? Yes, mechanically.)
plt.figure(figsize=(8, 6))
corr = df_final[['energy_100g', 'proteins_100g', 'carbohydrates_100g', 'sugars_100g', 'fat_100g']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Matrix of Nutritional Values', fontsize=16)
plt.show()

In [ ]:
# Visualization 3: Energy vs Fat Scatter Plot
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df_final, x='fat_100g', y='energy_100g', hue='product_name', s=100)
plt.title('Energy vs Fat Content', fontsize=16)
plt.xlabel('Fat (g per 100g)')
plt.ylabel('Energy (kJ per 100g)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

### Conclusion
By wrangling the data from disparate sources, cleaning missing values, and aggressively filtering physical impossibilities (outliers), we now have a highly structured baseline dataset that is ready for machine learning feature engineering (such as NLP on the `ingredients_text` column) to train our `nutrition-scorer` algorithms.